# N01 — Data Preparation (v12 pilot)

**Purpose**: Construct train/test split and apply an HC-specific sparse-binary
filter inline (this policy is data-specific to HC's FLAG_DOCUMENT_* columns;
not packaged). Single 80/20 split (pilot). CV executed in N_CV.

**Outputs** (`outputs/N01/`):
- `datasets.pkl`: {name: {X_train, X_test, y_train, y_test, feature_names, sparse_report}}
- `data_summary.csv`

Random state = 317 (frozen for data split reproducibility).

In [ ]:
import warnings; warnings.filterwarnings('ignore')
import pickle, numpy as np, pandas as pd
from pathlib import Path
from sklearn.model_selection import train_test_split
from decentra._utils import information_value

RS, TEST_SIZE = 317, 0.2
DATA_DIR = Path('../.data'); OUT = Path('../outputs/N01'); OUT.mkdir(parents=True, exist_ok=True)

def filter_sparse_binaries(X, y, min_rate=0.01, min_iv=0.02):
    '''Drop binary columns with minority<min_rate AND IV<min_iv. Data-specific
    policy; kept at notebook level rather than packaged.'''
    dropped, report = [], []
    for col in X.columns:
        vc = X[col].dropna().value_counts(normalize=True)
        if len(vc) != 2:
            continue
        minority = float(vc.min())
        iv = information_value(X[col].values, np.asarray(y))
        action = 'DROPPED' if (minority < min_rate and iv < min_iv) else 'KEPT'
        report.append({'column': col, 'minority_rate': minority, 'IV': iv, 'action': action})
        if action == 'DROPPED':
            dropped.append(col)
    return X.drop(columns=dropped), dropped, pd.DataFrame(report)

print('Ready')

## 1. Load and clean GMSC / HC

In [ ]:
# GMSC
df = pd.read_csv(DATA_DIR / 'give_me_some_credit/cs-training.csv', index_col=0)
y_gmsc = df['SeriousDlqin2yrs']
X_gmsc = df.drop('SeriousDlqin2yrs', axis=1).select_dtypes(include=[np.number]).copy()
X_gmsc['age'] = X_gmsc['age'].clip(18, 100)
print(f'GMSC: {X_gmsc.shape}, default {y_gmsc.mean():.1%}')

# HC
df = pd.read_csv(DATA_DIR / 'home_credit/application_train.csv')
y_hc = df['TARGET']
X_hc = df.drop(columns=['TARGET','SK_ID_CURR'], errors='ignore').select_dtypes(include=[np.number])
X_hc = X_hc[X_hc.columns[X_hc.isnull().mean() < 0.4]]
print(f'HC after miss>40% drop: {X_hc.shape}')

## 2. Split, impute, and apply `SparseBinaryFilter` per-dataset

The filter learns drop-rules from *train* (no test leakage) and applies the same
rules to test. In N_CV this reruns per-fold automatically.

In [ ]:
datasets = {}
for name, X, y in [('GMSC', X_gmsc, y_gmsc), ('HC', X_hc, y_hc)]:
    X_tr, X_te, y_tr, y_te = train_test_split(
        X, y, test_size=TEST_SIZE, stratify=y, random_state=RS)
    med = X_tr.median(); X_tr = X_tr.fillna(med); X_te = X_te.fillna(med)

    X_tr_f, dropped, report = filter_sparse_binaries(X_tr, y_tr)
    X_te_f = X_te.drop(columns=dropped)

    datasets[name] = {
        'X_train': X_tr_f, 'X_test': X_te_f,
        'y_train': y_tr, 'y_test': y_te,
        'feature_names': list(X_tr_f.columns),
        'train_medians': med, 'sparse_dropped': dropped,
        'sparse_report': report,
    }
    print(f'{name}: train={len(X_tr_f):,}, test={len(X_te_f):,}, features={X_tr_f.shape[1]}, '
          f'dropped={len(dropped)}')

## 3. Save

In [ ]:
with open(OUT/'datasets.pkl', 'wb') as f:
    pickle.dump(datasets, f)

summary = pd.DataFrame([{
    'Dataset': n, 'Train': len(d['X_train']), 'Test': len(d['X_test']),
    'Features': d['X_train'].shape[1], 'Default': f"{d['y_train'].mean():.1%}",
    'Dropped (sparse-binary)': len(d['sparse_filter'].dropped_columns_),
} for n, d in datasets.items()])
summary.to_csv(OUT/'data_summary.csv', index=False)
print(summary.to_string(index=False))